# MIRI Drug Repositioning Screen — xpert-fda-miri-v1

Finalized, author-aligned, L1000-contained FDA screen. Only FDA drugs present in the L1000 embedding table (exact SMILES match, drug_idx < 8981) are screened; the HG biological embeddings are the **author reference table** — no HG training / FDA-extension inference. Run cells in order. Environment: `conda activate arc`. Heavy steps (04 benchmark, 05 finetune, 06 predict) require a GPU.

## Step 01 — Drug list + embeddings

Filters the ChEMBL FDA table (`input/fda_drugs.csv`) to drugs whose exact canonical SMILES is in the L1000 rows of the embedding mapping (`drug_idx < 8981`), then copies the author reference HG table (workflow-owned `input/HG_drug_embeddings.npy`) to `01_drug_embed/output/all_drugs_idx2HG.npy` (byte-identical parity gate). Outputs: `01_drug_embed/output/drug_list.csv` (1,004 drugs) + `all_drugs_idx2HG.npy`.

In [ ]:
!python 01_drug_embed/drug_list.py
!python 01_drug_embed/drug_embeddings.py

## Gene mapping — authoritative DTOXS -> L1000 + verification

Builds the Entrez-verified DToxS->L1000 landmark gene map (resolves the 18 legacy-symbol landmarks such as WASHC5<-KIAA0196, JPT2<-HN1L) and gates that all 978 landmarks participate in the finetune. Run before step 02.

In [ ]:
!python _lib/verify_genes.py

## Step 02 — CM baseline transcriptome

Builds the consensus cardiomyocyte baseline from DTOXS control wells using the authoritative gene map (no lost legacy-symbol genes). Output: `02_cm_baseline/output/cm_ctl_landmark.csv`.

In [ ]:
!python 02_cm_baseline/cm_baseline.py

## Step 03 — XPert pretrain (reference, workflow-owned)

Holds the author pretrained XPert base weight (`pretrain_mdmt_full_50_epoch.pth`) and the L1000 pretrain YAML that produced it — the pretrain dataset lives under `pert_data/l1000-xpert/`. Not re-run during screening; steps 04/05 consume the weight from here.

In [ ]:
!ls 03_xpert_pretrain/

## Step 04 — Benchmark XPert on DTOXS splits

Regenerates the unified fold columns (warm-start + cold-drug, 10 folds each, ADR 0014) on the frozen `input/dtoxs.h5ad`, then finetunes and benchmarks each fold on its held-out test set (weights kept). The L1000 pretrain weight is also evaluated zero-shot (no finetuning) on all 20 fold test sets and plotted as the FIRST column (`pretrain`, amber). Per-fold values are overlaid as black dots on the mean ± std bars. Side branch: downstream steps use the step-05 model. Outputs: `04_xpert_finetune_bench/output/benchmark.csv` + `benchmark.png`.

In [ ]:
!python 04_xpert_finetune_bench/create_dtoxs_splits.py
!python 04_xpert_finetune_bench/xpert_finetune_bench.py

## Step 05 — Finetune XPert on DTOXS

Full-DToxS finetune -> the final prediction model (base weight from step 03). Idempotent (weights kept). Output: `05_xpert_finetune/output/dtoxs_dtoxs/`.

In [ ]:
!python 05_xpert_finetune/xpert_finetune.py

## Step 06 — Perturbed transcriptome + z-score

Predicts drug perturbation (deg) for the 1,004 L1000-contained FDA drugs via XPert (step-05 model), then z-score normalizes the DEG per gene across drugs. Predict indexes the precomputed UniMol array by drug index, so finetune and predict consume identical features. Outputs: `06_xpert_predict/output/xpert_fda-cm_deg.csv` + `-zs.csv`.

In [ ]:
!python 06_xpert_predict/perturb_transcriptome.py
!python 06_xpert_predict/zscore.py

## Step 07 — MIRI reference gene sets + pathways

Consumes the per-GSE DEG lists in `disease_data/miri/{GSE}/deg/` and produces the integrated reference (`MIRI_up-down.json`, `MIRI_rank.json`). `input/MIRI_pathways.json` is static (curated).

In [ ]:
!python 07_miri_phenotype/miri_phenotype.py

## Step 08 — Efficacy scores

Runs the three scorers (WCS+RGES, Zhang/ssCMap, pathway RGES) and computes the z-components for all screened drugs, plus the score-distribution plots. Output: `08_miri_efficacy_scores/output/scores.csv` + PNGs.

In [ ]:
!python 08_miri_efficacy_scores/miri_efficacy_scores.py

## Step 09 — Filter

Applies the WCS<0 and >=6/8 critical-pathway filters + filter-progression pyramid plot. Outputs: `09_filter/output/wcs_filtered.csv`, `path_filtered.csv`, `pyramid-filters.png`.

In [ ]:
!python 09_filter/filter.py

## Step 10 — Integrate

Computes the integrated score over all screened drugs, ranks the filtered candidates, and writes the final report: `10_integrate/output/integrated_scores.csv`, `ranked_drugs.csv`, `report.md` + rank heatmaps.

In [ ]:
!python 10_integrate/integrate.py

## Step 11 — Final bilingual drug report

Agentic per-drug fact sheets (biomni A1 backend, external dependency) over the final drugs (`09_filter` survivors with integrated percentile > 0.5), assembled into `11_report/output/report.md` (EN) + `report_cn.md` (CN). Resumable via per-drug JSON cache.

In [ ]:
!python 11_report/run_drug_agents.py
!python 11_report/assemble_report.py
!python 11_report/assemble_cn_report.py

## Pipeline summary

The final ranked drug list is `10_integrate/output/ranked_drugs.csv` with the deterministic report at `10_integrate/output/report.md` and the agentic drug report at `11_report/output/report.md`.

In [ ]:
import pandas as pd
ranked = pd.read_csv('10_integrate/output/ranked_drugs.csv')
print(f'{len(ranked)} drugs passed the filters')
print(ranked[['drug_name', 'integrated']].head(10).to_string(index=False))